# ELEC3305 - Real Time Sonar App


In [1]:
# Import functions and libraries
%gui qt

import numpy as np
from scipy import signal
from rtsonar import rtsonar

In [ ]:
import pyaudio
p = pyaudio.PyAudio()
for i in range(p.get_device_count()):
    info = p.get_device_info_by_index(i)
    if info['maxInputChannels'] > 0:
        print(f"Index {i}: {info['name']}")
p.terminate()

### Task III: Play with different parameters with the real-time sonar!

#### 15 Marks

The function `rtsonar()` in `rtsonar.py` provides a wrapper to create a real-time sonar plot using user-defined processing functions. Copy and paste your functions genPulseTrain(), genChirpPulse(), crossCorr(), findDelay(), and dist2time() to the code cell below. Scroll down to the bottom and you should be able to run `rtsonar()`!

The detection resolution and sensitivity will depend on the parameters you will choose for the sonar. The longer the pulse is, the worse time resolution you will get, but much stronger matched filtering amplitude. You can buy the time resolution by using pulse compression with chirp pulses at the expense of increasing the bandwidth. 

* Run the sonar with different parameters. You can get interesting targets by moving your laptop, moving yourself next to the computer, using a book as a reflecting surface. It would be easier for you to distinguish the target if you move it back and forth. Play with the pulse lengths, the frequency sweeps, etc. 
* To get the best real-time performance, you should restart the kernel every time you run this lab. 
* Submit 4 interesting set of parameters. Explain why you chose them. 




In [2]:
def genChirpPulse(Npulse, f0, f1, fs):
    # Calculate time based on number of samples
    t = np.arange(int(Npulse)) / fs
    T = Npulse / fs
    # Generate complex analytic chirp
    pulse = np.exp(1j * 2 * np.pi * (f0 * t + ((f1 - f0) / (2 * T)) * (t**2)))
    # Reshape to (Npulse, 1) to match the wrapper's Hanning window application
    return pulse.reshape(-1, 1)

def genPulseTrain(pulse, Nrep, Nseg):
    # Flatten to 1D and pad with zeros
    pulse = np.ravel(pulse)
    pulse_padded = np.zeros(int(Nseg))
    pulse_padded[:len(pulse)] = pulse
    return np.tile(pulse_padded, int(Nrep))

def crossCorr(rcv, pulse_a):
    # Ensure both inputs are 1D arrays for convolution
    rcv = np.ravel(rcv)
    pulse_a = np.ravel(pulse_a)
    return signal.fftconvolve(rcv, np.conj(pulse_a)[::-1], mode='full')

def findDelay(Xrcv, Nseg):
    # Return peak index as a standard Python integer
    return int(np.argmax(np.abs(np.ravel(Xrcv)[:int(Nseg)])))

def dist2time(dist, temperature=21):
    # Speed of sound formula
    speed_of_sound = 331.3 * np.sqrt(1 + temperature / 273.15)
    # 2x for round trip, distance in cm to m
    time_val = 2 * (float(dist) / 100.0) / speed_of_sound
    return float(time_val)

In [ ]:
# --- SONAR PARAMETERS ---
fs = 48000 # Sampling frequency
f0 = 6000 # Chirp initial frequency
f1 = 12000 # Chirp ending frequency

Npulse = 500 # Length of Chirp Pulse
Nrep = 24 # Number of repetition in a pulse train (determines vertical length of plot )
Nseg = 2048*2 # Number of samples between pulses (determines maximum time-of-arrival)
Nplot = 200 # Number of pixels in plot along horizontal direction (higher resolution is slower)
maxdist = 200 # Maximum distance in cm
temperature = 20 # Temperature     

# Join the functions together
functions = (genChirpPulse, genPulseTrain, crossCorr, findDelay, dist2time)

stop_flag = rtsonar( f0, f1, fs, Npulse, Nseg, Nrep, Nplot, maxdist, temperature, functions, input_device_index=0)

In [ ]:
# Run this to stop the sonar (it will take a few seconds to stop)
stop_flag.set()